# 🧼 Milestone 2: LinkedIn Profile Cleaning Notebook
## Project: Career Pathways into IT for Displaced Youth (U.S., 2025)
**Group:** 4  
**Author:** Yuri Spizhovyi  
**Source:** [Milestone 2 – Data Preparation](https://github.com/MIT-Emerging-Talent/ET6-CDSP-group-04-repo/tree/main/2_data_preparation)  
**Repo:** [My Repo](https://github.com/MIT-Emerging-Talent/ET6-CDSP-group-04-repo)

---

This notebook processes and cleans data extracted from 370+ LinkedIn profiles of IT professionals in the United States. The raw profiles were scraped and parsed using custom Playwright and BeautifulSoup scripts.

We extract relevant fields related to education, employment, and experience, and engineer new features to support downstream analysis, such as:

- **Has_Degree**: Whether the professional lists any post-secondary degree
- **Education_Type**: Categorized as `University`, `Bootcamp`, `Online`, or `Unknown`
- **Job_Category**: Grouped job role (e.g., `Software Engineer`, `QA`, `DevOps`)
- **Years_Experience**: Estimated years based on first job start and end dates

This dataset allows us to explore the relationship between educational background and employment outcomes in IT roles — especially to assess how common alternative education models are among working professionals.


In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
# Clone once per session
!git clone https://github.com/MIT-Emerging-Talent/ET6-CDSP-group-04-repo.git

# Move into the correct folder
%cd ET6-CDSP-group-04-repo
#%cd /content/

fatal: destination path 'ET6-CDSP-group-04-repo' already exists and is not an empty directory.
/content


## Load Raw Data

In [64]:
#df = pd.read_csv("/content/raw_parsed_linkedin_output.csv")
df = pd.read_csv("1_dataset/raw_data/raw_parsed_linkedin_output.csv")
df.head()

,File,Scrape_ID,Scrape_Date,Name,Location,Experience_1,Company_1,Date_1_start,Date_1_end,Education_1,Degree,Field,Education_Start,Education_End,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17
0,Aaron_Baczek_20250713_192041.html,8fc8672e-2cf3-4011-b9c5-f6916fc05832,2025-07-15,Aaron Baczek,"Louisville, Colorado, United States",Retired/Contract Automation Project Manager,Dickerson Contracting · Full-time,2003,2018,John A. Logan College,Second degree connection,Keith A Harpool Engineering Consulting LLC,2000,2003,NaN,NaN,NaN,NaN
1,Aaron_Taylor_20250714_231807.html,24ef40fd-4c99-4499-ba48-d6d25e2d120e,2025-07-15,Aaron Taylor,"Federal Way, Washington, United States",Software Engineer,"Pacific Software Publishing, Inc. · Full-time",2019,Present,NaN,Third degree connection,Director of Engineering at Infinite Red,2019,2019,NaN,NaN,NaN,NaN
2,Abhigyan_Sinha_20250714_234649.html,26945ffe-d349-4cda-a7a5-8bcabea48ab8,2025-07-15,Abhigyan Sinha,Denver Metropolitan Area,Frontend Engineer,Full-time,2019,2021,Staff Software Engineer at Visa| Backend Engin...,Third degree connection,CEO @ DataLemur (FAANG SQL Interview Prep) • E...,2013,2017,NaN,NaN,NaN,NaN
3,Abhigyan_Sinha_20250715_000127.html,75f5f5b7-4175-4ffd-8dc3-d0eb0963a83d,2025-07-15,Abhigyan Sinha,Denver Metropolitan Area,Software Engineer,Full-time,2019,2021,Staff Software Engineer at Visa| Backend Engin...,Third degree connection,CEO @ DataLemur (FAANG SQL Interview Prep) • E...,2013,2017,NaN,NaN,NaN,NaN
4,Abhijit_Tambe_20250713_192312.html,97b9c558-3265-4864-a4c6-0dd474c61c1f,2025-07-15,Abhijit Tambe,"McLean, Virginia, United States",Deputy Manager of Private Client Advisory Serv...,Marlabs LLC · Full-time,2015,2018,"Student at Pace University<span class=""white-s...",Second degree connection,Sales Associate | Social Media Content Creator...,2015,2018,NaN,NaN,NaN,NaN


### 📝 Observations & Next Steps

From the initial inspection, we observed:

- All expected columns are present, but some contain inconsistent or incomplete data:
  - `Degree` and `Education_1` contain placeholder values like "N/A" or are blank in many rows
  - Job titles in `Experience_1` vary widely and use mixed formatting
  - Dates in `Date_1_start` and `Date_1_end` are not standardized and often contain the word "Present"
- Some fields extracted from the HTML (e.g., `Field`) are noisy or misclassified
- Importantly, **a missing `Education_1` field does not mean a person lacks education**. In the IT industry, it's more realistic to assume such individuals gained skills through **online platforms or informal training** — not that they received "no education"

🛠️ **Based on these findings, we decided to:**
- Create a new binary column `Has_Degree` based on the presence of formal degree information
- Engineer a new `Education_Type` column using keyword matching to classify entries as:
  - `"University"` – traditional higher ed
  - `"Bootcamp"` – immersive, short-term programs
  - `"Online/Courses"` – online platforms or self-taught learners (also used for blank entries)
- Group job titles into simplified categories via a new `Job_Category` field
- Estimate work experience using start/end years to compute `Years_Experience`
- Clean and filter the dataset to allow for structured analysis in Milestone 3

This preparation ensures that we can meaningfully compare employment outcomes across different educational backgrounds in the IT field — including both traditional and alternative learning pathways.


In [65]:
df.info()
df.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 377 entries, 0 to 376
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   File             377 non-null    object 
 1   Scrape_ID        377 non-null    object 
 2   Scrape_Date      377 non-null    object 
 3   Name             377 non-null    object 
 4   Location         370 non-null    object 
 5   Experience_1     370 non-null    object 
 6   Company_1        322 non-null    object 
 7   Date_1_start     354 non-null    object 
 8   Date_1_end       365 non-null    object 
 9   Education_1      351 non-null    object 
 10  Degree           361 non-null    object 
 11  Field            308 non-null    object 
 12  Education_Start  358 non-null    object 
 13  Education_End    359 non-null    object 
 14  Unnamed: 14      62 non-null     object 
 15  Unnamed: 15      19 non-null     object 
 16  Unnamed: 16      6 non-null      float64
 17  Unnamed: 17     

,File,Scrape_ID,Scrape_Date,Name,Location,Experience_1,Company_1,Date_1_start,Date_1_end,Education_1,Degree,Field,Education_Start,Education_End,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17
count,377,377,377,377,370,370,322,354,365,351,361,308,358,359,62,19,6.000000,2.000000
unique,377,377,1,300,124,184,209,37,27,198,51,197,78,43,22,10,NaN,NaN
top,Zachary_Kupu_20250714_230941.html,aa6094b2-f95d-416a-9a91-0b629ed28dcc,2025-07-15,Sevilay Ozturk,United States,Software Engineer,Full-time,2019,Present,Student at The University of British Columbia,Second degree connection,Second degree connection,2015,2022,2020,2011,NaN,NaN
freq,1,1,377,5,57,20,33,33,62,68,157,21,27,31,6,4,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020.166667,2017.000000
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.013873,7.071068
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008.000000,2012.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021.500000,2014.500000
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023.000000,2017.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023.000000,2019.500000


### 1️⃣ Dropped Unnecessary Columns
The raw CSV contained extra columns such as `Unnamed: 14–17` which were likely remnants from a misaligned or auto-expanded CSV export. These were dropped to clean the structure and simplify further processing.
### 🧹 Removing Incomplete Profiles with No Experience or Education Data

In addition to placeholder rows like `"Welcome to your professional community"`, we identified profiles where:

- The `Name` and `Location` fields are filled
- But **all columns after `Location` are blank or missing**

These rows contain no usable information on experience or education, so we removed them using the logic below.


In [66]:
# Step 1: Drop unnecessary columns
df_cleaned = df.drop(columns=[col for col in df.columns if "Unnamed" in col])

# Step 2: Drop rows where all columns after 'Location' are empty
cols_after_location = df.columns.tolist()[df.columns.get_loc("Location") + 1:]
df = df.dropna(subset=cols_after_location, how="all")

# Step 3: Drop duplicate profiles based on 'Name'
df = df.drop_duplicates(subset=["Name"], keep="first")

# Display how many rows remain and example duplicates removed
df.shape, df["Name"].value_counts().head(10)

((294, 18),
 Name
 Zachary Kupu            1
 Aaron Baczek            1
 Aaron Taylor            1
 Abhigyan Sinha          1
 Abhijit Tambe           1
 Abhilash Reddy          1
 Vitaliy Prysiazhniuk    1
 Vishnu K.               1
 Violetta Malackowski    1
 Vinod R                 1
 Name: count, dtype: int64)

### 2️⃣ Created `Has_Degree` Column

We introduced a new binary feature `Has_Degree` to indicate whether a LinkedIn profile lists a formal educational qualification.

- A value of `1` means the profile contains non-empty and meaningful content in the `Degree` field (excluding "N/A" or missing).
- A value of `0` means no degree information was found or the field was explicitly marked as "N/A".

This feature helps us differentiate between IT professionals who have obtained formal degrees versus those who may have entered the field through non-traditional or alternative pathways.
*italicized text*

In [67]:
df["Has_Degree"] = df["Degree"].apply(lambda x: 0 if pd.isna(x) or x.strip().lower() == "n/a" else 1)

### 3️⃣ Classified `Education_Type`

We created a new categorical feature `Education_Type` to represent the type of educational background each LinkedIn profile reflects. This classification is based on keyword detection in the `Education_1` field and aligned with practical assumptions in the IT industry.

The categories are defined as:

- **University** – if the entry includes terms like "university", "college", "institute", "academy", or similar
- **Bootcamp** – if known bootcamp names such as "Hack Reactor", "General Assembly", or "Code Institute" are found (none detected in this dataset)
- **Online/Courses** – if the field is empty, contains "N/A", or includes terms like "Coursera", "edX", "Udemy", "freeCodeCamp", or "online"
  
> In the IT sector, it’s highly unlikely for someone to work in a technical role with zero training. Therefore, missing or generic entries are assumed to indicate non-traditional or online learning rather than a complete lack of education.

This feature enables us to explore how different educational pathways relate to IT employment outcomes, particularly in comparing formal degrees to alternative training models.


In [68]:
# Step 3: Create "Education_Type" from "Education_1"
def classify_education_type_fixed(text):
    if pd.isna(text) or text.strip() == "":
        return "Online/Courses"
    text = text.lower()
    if any(k in text for k in ["bootcamp", "hack reactor", "general assembly", "code institute"]):
        return "Bootcamp"
    elif any(k in text for k in ["coursera", "udemy", "udacity", "edx", "freecodecamp", "online"]):
        return "Online/Courses"
    elif any(k in text for k in ["university", "college", "institute", "school", "academy", "polytechnic"]):
        return "University"
    else:
        return "Online/Courses"

df["Education_Type"] = df["Education_1"].apply(classify_education_type_fixed)


### 4️⃣ Categorized Job Titles with `Job_Category`

We added a new feature `Job_Category` to simplify the wide variety of job titles found in the `Experience_1` column. This classification allows us to group similar roles together for easier analysis and visualization.

The job titles were categorized into the following buckets based on keyword patterns:

- **Software Engineer** – includes roles with "engineer" or "developer"
- **Data** – includes roles with "data" such as "Data Analyst", "Data Scientist"
- **DevOps** – includes mentions of "DevOps" or related tooling roles
- **QA** – includes "QA", "Quality Assurance", or testing roles
- **Manager** – includes management or leadership titles with "Manager"
- **Other** – any other roles not falling into the categories above

In [69]:
# Step 4: Categorize job title
def classify_job(title):
    if pd.isna(title):
        return "Other"
    title = title.lower()
    if "data" in title:
        return "Data"
    elif "qa" in title or "quality" in title:
        return "QA"
    elif "devops" in title:
        return "DevOps"
    elif "engineer" in title or "developer" in title:
        return "Software Engineer"
    elif "manager" in title:
        return "Manager"
    else:
        return "Other"

df["Job_Category"] = df["Experience_1"].apply(classify_job)


### 5️⃣ Estimated `Years_Experience`

To approximate each individual's time in the IT workforce, we created a new numeric feature: `Years_Experience`. This is calculated using the `Date_1_start` and `Date_1_end` fields, which represent the start and end years of the professional's first listed job.

- If the `Date_1_end` value is `"Present"` (or contains the word), we assume the profile is active as of **2025**
- If either date is missing or non-numeric, the result is set to `NaN`
- Only positive or realistic durations are retained for analysis

This feature allows us to evaluate experience trends across education types and job categories.

In [70]:
def calculate_years(start, end):
    try:
        start_year = int(start)
        end_year = int(end) if end.lower() != "present" else 2025
        return end_year - start_year if end_year >= start_year else np.nan
    except:
        return np.nan

df["Years_Experience"] = df.apply(
    lambda row: calculate_years(str(row["Date_1_start"]), str(row["Date_1_end"])), axis=1
)

### 🧼 Fixing Inconsistent Date Fields

Several columns in our LinkedIn dataset — `Date_1_start`, `Date_1_end`, `Education_Start`, and `Education_End` — contained inconsistent or non-numeric values. These included:

- Strings like `"Present"`, which we interpret as currently employed or enrolled (mapped to the year **2025**)
- Descriptive text like `"🎮 Computer Science Student..."` or company roles
- Garbled or impossible numeric values (e.g., `"1285"`)

To standardize these columns for analysis and modeling, we defined a helper function `clean_year()` that extracts the first valid 4-digit year or handles `"Present"` cases.

In [71]:
import re
import numpy as np

def clean_year(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if "present" in value.lower():
        return 2025
    match = re.search(r"\b(19|20)\d{2}\b", value)
    return int(match.group()) if match else np.nan

for col in ["Date_1_start", "Date_1_end", "Education_Start", "Education_End"]:
    df[col] = df[col].apply(clean_year)


## 💾 Final Output: Cleaned LinkedIn Dataset

After completing all cleaning and feature engineering steps, we saved the final version of the dataset as a CSV file named:



In [72]:
#### ✅ Save to CSV
df.to_csv("cleaned_linkedin_profiles.csv", index=False)
print("✅ Cleaned data saved as cleaned_linkedin_profiles.csv")

✅ Cleaned data saved as linkedin_cleaned.csv



This file includes the following core columns:

| Column Name         | Description |
|---------------------|-------------|
| `Name`              | Full name from LinkedIn profile |
| `Location`          | Self-reported location |
| `Experience_1`      | Most recent or prominent job title |
| `Company_1`         | Associated company |
| `Date_1_start`      | Job start year |
| `Date_1_end`        | Job end year or "Present" |
| `Education_1`       | Name of educational institution |
| `Degree`            | Degree title if available |
| `Field`             | Field of study or specialization |
| `Education_Start`   | Education start year |
| `Education_End`     | Education end year |
| `Has_Degree`        | Binary (1 = degree info present, 0 = not present) |
| `Education_Type`    | University / Bootcamp / Online/Courses |
| `Job_Category`      | Grouped job type (e.g., Software Engineer, QA) |
| `Years_Experience`  | Estimated duration of professional experience |

This cleaned dataset will be used in **Milestone 3: Data Analysis** to examine employment outcomes across different education pathways.